In [4]:
using Random, Distributions, Statistics
using Convex, MosekTools, LinearAlgebra

# ---------- mixture grid + fit (unchanged) ----------
function default_grid_for_U(U::Vector{Float64}; n_mu=20, n_sigma=50)
    qlo, qhi = quantile(U, (0.0001, 0.9999))
    pad = 0.2 * (qhi - qlo)
    mus = collect(range(qlo - pad, qhi + pad; length=n_mu))

    sU = std(U)
    σ_lo = max(1e-3, 0.05 * sU)
    σ_hi = max(σ_lo * 2, 2.0 * sU)
    sigmas = exp.(range(log(σ_lo), log(σ_hi); length=n_sigma))

    return mus, sigmas
end

function fit_fixedgrid_gaussian_mixture_mosek(U::Vector{Float64};
    mus::Vector{Float64},
    sigmas::Vector{Float64},
    verbose::Bool=false
)
    N = length(U)

    mu_list = Float64[]
    sig_list = Float64[]
    for μ in mus, σ in sigmas
        push!(mu_list, μ)
        push!(sig_list, σ)
    end
    K = length(mu_list)

    logφ = Matrix{Float64}(undef, N, K)
    @inbounds for j in 1:K
        dj = Normal(mu_list[j], sig_list[j])
        for i in 1:N
            logφ[i, j] = logpdf(dj, U[i])
        end
    end

    m = vec(maximum(logφ, dims=2))
    A = exp.(clamp.(logφ .- m, -745.0, 0.0))

    π = Variable(K)
    constraints = [π >= 0, sum(π) == 1]
    obj = sum(m) + sum(log(A * π))
    problem = maximize(obj, constraints)

    solve!(problem, Mosek.Optimizer; silent=!verbose)

    π_hat = vec(evaluate(π))
    π_hat = max.(π_hat, 0.0)
    π_hat ./= sum(π_hat)

    comps = (mu = mu_list, sigma = sig_list)
    return π_hat, comps
end

# ---------- helper: unbiased SD ----------
@inline function sd_unbiased(x)
    m = length(x)
    @assert m ≥ 2 "need at least 2 points for SD"
    μ = mean(x)
    return sqrt(sum((x .- μ).^2) / (m - 1))
end

# ---------- NEW for n=3: drop the point farthest from the median ----------
@inline function drop_farthest_from_median(x::AbstractVector{<:Real})
    n = length(x)
    @assert n == 3 "this helper is intended for n=3"

    xs = sort(collect(x))
    med = xs[2]

    dleft  = med - xs[1]
    dright = xs[3] - med

    if dleft > dright
        return @view xs[2:3]   # drop left extreme
    elseif dright > dleft
        return @view xs[1:2]   # drop right extreme
    else
        return @view xs[1:2]   # deterministic tie-break
    end
end

@inline drop1_mean(x) = mean(drop_farthest_from_median(x))

@inline function drop1_sd(x)
    y = drop_farthest_from_median(x)
    return sd_unbiased(y)
end

"""
Simulate B datasets of size n=3 from N(0,1), return:
  T = mean after dropping the point farthest from the median
  S = SD   after dropping the point farthest from the median
  U = log(S)
"""
function simulate_drop1mean_drop1SD_U_n3(B::Int; rng=Random.default_rng())
    n = 3
    T = Vector{Float64}(undef, B)
    S = Vector{Float64}(undef, B)
    x = Vector{Float64}(undef, n)
    dist = Normal(0,1)

    @inbounds for b in 1:B
        rand!(rng, dist, x)
        T[b] = drop1_mean(x)
        S[b] = drop1_sd(x)
    end

    U = log.(S .+ 1e-300)
    return T, S, U
end



simulate_drop1mean_drop1SD_U_n3

In [5]:
# ---------- example run for n=3 ----------
rng = MersenneTwister(3)
# B = 600_000
B = 600_000

T_drop1, S_drop1, U_drop1 = simulate_drop1mean_drop1SD_U_n3(B; rng=rng)

mus, sigmas = default_grid_for_U(U_drop1; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U_drop1; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))

[ Info: [Convex.jl] Compilation finished: 2358.91 seconds, 150.474 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1001            
  Affine conic cons.     : 600000 (1800000 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 601000          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 1.15            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0      

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124


done: K = 1000, sum(π)= 1.0


In [8]:
n=3

3

In [9]:
using JLD2

# after fitting:
@save "U_mixture_fit_1.3_trimSD(n=3).jld2" π_hat comps n B


In [7]:
n

LoadError: UndefVarError: `n` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
using Random, Distributions, Statistics
using Convex, MosekTools, LinearAlgebra

# ---------- mixture grid + fit (unchanged) ----------
function default_grid_for_U(U::Vector{Float64}; n_mu=20, n_sigma=50)
    qlo, qhi = quantile(U, (0.0001, 0.9999))
    pad = 0.2 * (qhi - qlo)
    mus = collect(range(qlo - pad, qhi + pad; length=n_mu))

    sU = std(U)
    σ_lo = max(1e-3, 0.05 * sU)
    σ_hi = max(σ_lo * 2, 2.0 * sU)
    sigmas = exp.(range(log(σ_lo), log(σ_hi); length=n_sigma))

    return mus, sigmas
end

function fit_fixedgrid_gaussian_mixture_mosek(U::Vector{Float64};
    mus::Vector{Float64},
    sigmas::Vector{Float64},
    verbose::Bool=false
)
    N = length(U)

    mu_list = Float64[]
    sig_list = Float64[]
    for μ in mus, σ in sigmas
        push!(mu_list, μ)
        push!(sig_list, σ)
    end
    K = length(mu_list)

    logφ = Matrix{Float64}(undef, N, K)
    @inbounds for j in 1:K
        dj = Normal(mu_list[j], sig_list[j])
        for i in 1:N
            logφ[i, j] = logpdf(dj, U[i])
        end
    end

    m = vec(maximum(logφ, dims=2))
    A = exp.(clamp.(logφ .- m, -745.0, 0.0))

    π = Variable(K)
    constraints = [π >= 0, sum(π) == 1]
    obj = sum(m) + sum(log(A * π))
    problem = maximize(obj, constraints)

    solve!(problem, Mosek.Optimizer; silent=!verbose)

    π_hat = vec(evaluate(π))
    π_hat = max.(π_hat, 0.0)
    π_hat ./= sum(π_hat)

    comps = (mu = mu_list, sigma = sig_list)
    return π_hat, comps
end

# # ---------- NEW: median + MAD about median ----------
# @inline mad_median(x) = median(abs.(x .- median(x)))   # MAD about the median

# """
# Simulate B datasets of size n from N(0,1), return:
#   T  = median(x)
#   D  = MAD about median = median(|x - median(x)|)
#   U  = log(D)
# """
# function simulate_median_MADmed_U(n::Int, B::Int; rng=Random.default_rng())
#     T = Vector{Float64}(undef, B)   # median
#     D = Vector{Float64}(undef, B)   # MAD about median
#     x = Vector{Float64}(undef, n)
#     dist = Normal(0,1)

#     @inbounds for b in 1:B
#         rand!(rng, dist, x)
#         med = median(x)
#         T[b] = med
#         D[b] = median(abs.(x .- med))
#     end

#     U = log.(D .+ 1e-300)
#     return T, D, U
# end

In [ ]:
# ---------- NEW: option 4 helpers (drop min/max) ----------

# Return a view/copy of the trimmed vector x_{(2)},...,x_{(n-1)}.
# Uses sorting (fine for small n; robust + simple).
@inline function trim_drop_minmax(x::AbstractVector{<:Real})
    n = length(x)
    @assert n ≥ 3 "need n ≥ 3 to drop min/max"
    xs = sort(collect(x))
    return @view xs[2:end-1]   # length n-2
end

# trimmed mean
@inline trim_mean(x) = mean(trim_drop_minmax(x))

# trimmed SD with Bessel correction on trimmed sample:
# sd_trim = sqrt( sum((y - mean(y))^2)/(m-1) ) where m = n-2
@inline function trim_sd(x)
    y = trim_drop_minmax(x)
    m = length(y)
    @assert m ≥ 2 "need at least 2 points after trimming (n ≥ 4) to compute SD"
    μ = mean(y)
    s2 = sum((y .- μ).^2) / (m - 1)
    return sqrt(s2)
end

"""
Simulate B datasets of size n from N(0,1), return:
  T  = trimmed mean (drop min/max)
  S  = trimmed SD   (drop min/max, Bessel-corrected on m=n-2)
  U  = log(S)
"""
function simulate_trimmean_trimSD_U(n::Int, B::Int; rng=Random.default_rng())
    @assert n ≥ 4 "option 4 needs n ≥ 4 (after dropping 2 points, m=n-2 must be ≥ 2)"
    T = Vector{Float64}(undef, B)
    S = Vector{Float64}(undef, B)
    x = Vector{Float64}(undef, n)
    dist = Normal(0,1)

    @inbounds for b in 1:B
        rand!(rng, dist, x)
        tm = trim_mean(x)
        ts = trim_sd(x)
        T[b] = tm
        S[b] = ts
    end

    U = log.(S .+ 1e-300)
    return T, S, U
end

In [ ]:
# ---------- example run: option 4 likelihood on log(trimmed SD) ----------
rng = MersenneTwister(3)
n = 3
B = 600_000

T_trim, S_trim, U_trim = simulate_trimmean_trimSD_U(n, B; rng=rng)

mus, sigmas = default_grid_for_U(U_trim; n_mu=20, n_sigma=50)
π_hat, comps = fit_fixedgrid_gaussian_mixture_mosek(U_trim; mus=mus, sigmas=sigmas, verbose=true)

println("done: K = ", length(π_hat), ", sum(π)= ", sum(π_hat))